# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

> NOTE: DO NOT RUN THESE CELLS IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY

In [20]:
#!pip install -qU ragas==0.2.10

In [21]:
#!pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 unstructured==0.16.12 langgraph==0.2.61 langchain-qdrant==0.2.0

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [22]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /home/saimo/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/saimo/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [23]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [24]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [25]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Loan Data use-case!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [26]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [27]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [28]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [29]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs[:20]:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 20, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [30]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/18 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/29 [00:00<?, ?it/s]

Property 'summary' already exists in node 'df7288'. Skipping!
Property 'summary' already exists in node '68719f'. Skipping!
Property 'summary' already exists in node 'd00e9e'. Skipping!
Property 'summary' already exists in node '58b913'. Skipping!
Property 'summary' already exists in node '539412'. Skipping!
Property 'summary' already exists in node 'a8ba39'. Skipping!
Property 'summary' already exists in node '09e852'. Skipping!
Property 'summary' already exists in node '78c27d'. Skipping!
Property 'summary' already exists in node 'e8db1d'. Skipping!
Property 'summary' already exists in node '0dcf01'. Skipping!
Property 'summary' already exists in node 'dedfd3'. Skipping!


Applying CustomNodeFilter:   0%|          | 0/14 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/53 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node 'df7288'. Skipping!
Property 'summary_embedding' already exists in node 'd00e9e'. Skipping!
Property 'summary_embedding' already exists in node '58b913'. Skipping!
Property 'summary_embedding' already exists in node '0dcf01'. Skipping!
Property 'summary_embedding' already exists in node '539412'. Skipping!
Property 'summary_embedding' already exists in node 'dedfd3'. Skipping!
Property 'summary_embedding' already exists in node '78c27d'. Skipping!
Property 'summary_embedding' already exists in node '09e852'. Skipping!
Property 'summary_embedding' already exists in node '68719f'. Skipping!
Property 'summary_embedding' already exists in node 'e8db1d'. Skipping!
Property 'summary_embedding' already exists in node 'a8ba39'. Skipping!


Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 43, relationships: 433)

We can save and load our knowledge graphs as follows.

In [31]:
kg.save("loan_data_kg.json")
loan_data_kg = KnowledgeGraph.load("loan_data_kg.json")
loan_data_kg

KnowledgeGraph(nodes: 43, relationships: 433)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [32]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=loan_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [33]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.


####  Answer ✅ :

The three types of query synthesizers are:

1. SingleHopSpecificQuerySynthesizer

    🔹 Generates direct, fact-based questions that can be answered using a single piece of information from the data.

    🔹 Example: “What is the interest rate for a Direct Subsidized Loan?”

2. MultiHopAbstractQuerySynthesizer

    🔹 Creates more abstract questions that require connecting information from multiple sources and often involve reasoning or summarization.

    🔹 Example: “How is privacy maintained throughout the loan process?”

3. MultiHopSpecificQuerySynthesizer

    🔹 Produces specific, fact-based questions that require gathering and linking facts from multiple places in the data, but the answer is still concrete.
    
    🔹 Example: “How does the Department verify the student's SSN and what documents are involved in this process?”


In simple terms:

1. SingleHopSpecific = One-step, direct questions.

2. MultiHopAbstract  = Multi-step, reasoning or summary questions.

3. MultiHopSpecific  = Multi-step, but still concrete/fact-based questions.

Finally, we can use our `TestSetGenerator` to generate our testset!

In [34]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,How does the IRS influence the federal student...,[Application and Verification Guide Introducti...,The IRS is involved through the Fostering Unde...,single_hop_specifc_query_synthesizer
1,Has the FAFSA renewal functionality been inclu...,[Chapter 1: The Application Process We removed...,We removed the <Returning FAFSA Filers= sectio...,single_hop_specifc_query_synthesizer
2,How do FAFSA Partner Portal help with student ...,[The FPS also checks the application for possi...,The FAFSA Partner Portal allows schools to req...,single_hop_specifc_query_synthesizer
3,Could you please explain what constitutes a Va...,[Valid Output Document 34 CFR 668.2(b)],A Valid Output Document is defined in 34 CFR 6...,single_hop_specifc_query_synthesizer
4,What is Electronic Announcement GENERAL-23-34 ...,[2. The disclosure of their FTI by the IRS to ...,Electronic Announcement GENERAL-23-34 provides...,single_hop_specifc_query_synthesizer
5,How do discharge conditions and legal emancipa...,[<1-hop>\n\nVeteran of the U.S. Armed Forces T...,"Discharge conditions, such as being a veteran ...",multi_hop_abstract_query_synthesizer
6,How do discharge conditions and dependency cri...,[<1-hop>\n\nVeteran of the U.S. Armed Forces T...,"Discharge conditions, such as being discharged...",multi_hop_abstract_query_synthesizer
7,How do living arrangements and support criteri...,"[<1-hop>\n\nveteran, or will be one by June 30...","According to the provided context, a student c...",multi_hop_abstract_query_synthesizer
8,"VA veteran or not VA, how does that affect ind...","[<1-hop>\n\nveteran, or will be one by June 30...",The context explains that if a student provide...,multi_hop_specific_query_synthesizer
9,How does the FSA guide assist financial aid ad...,[<1-hop>\n\nApplication and Verification Guide...,The FSA guide provides detailed instructions f...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [35]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs[:20], testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/18 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/27 [00:00<?, ?it/s]

Property 'summary' already exists in node 'eca422'. Skipping!
Property 'summary' already exists in node 'c46ccf'. Skipping!
Property 'summary' already exists in node 'c2adb3'. Skipping!
Property 'summary' already exists in node 'b397f1'. Skipping!
Property 'summary' already exists in node 'd5e515'. Skipping!
Property 'summary' already exists in node '522de1'. Skipping!
Property 'summary' already exists in node 'feecd4'. Skipping!
Property 'summary' already exists in node '61cb8e'. Skipping!
Property 'summary' already exists in node '6eebae'. Skipping!


...

Property 'summary_embedding' already exists in node 'b397f1'. Skipping!
Property 'summary_embedding' already exists in node 'c2adb3'. Skipping!
Property 'summary_embedding' already exists in node 'feecd4'. Skipping!
Property 'summary_embedding' already exists in node 'c46ccf'. Skipping!
Property 'summary_embedding' already exists in node '522de1'. Skipping!
Property 'summary_embedding' already exists in node 'd5e515'. Skipping!
Property 'summary_embedding' already exists in node '61cb8e'. Skipping!
Property 'summary_embedding' already exists in node '6eebae'. Skipping!
Property 'summary_embedding' already exists in node 'eca422'. Skipping!


Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [36]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,Could you explain the significance of ED in th...,[Application and Verification Guide Introducti...,"In the context of the provided guide, ED refer...",single_hop_specifc_query_synthesizer
1,What is the FSA KNowledge Center?,[electronically due to limitations on access t...,The FSA Knowledge Center is a resource page th...,single_hop_specifc_query_synthesizer
2,What are the differences between the 2024-25 F...,[Note: There will be two active portals that c...,There are two active portals corresponding to ...,single_hop_specifc_query_synthesizer
3,Could you please explain the significance of t...,[Output Documents After processing is complete...,"The 2025-26 FAFSA Specifications Guide, specif...",single_hop_specifc_query_synthesizer
4,How does the IRS data about tax info relate to...,[<1-hop>\n\n2. The disclosure of their FTI by ...,"The IRS data, considered FTI, includes informa...",multi_hop_abstract_query_synthesizer
5,How do mailing procedures and address changes ...,[<1-hop>\n\nelectronically due to limitations ...,Mailing procedures for FAFSA forms specify tha...,multi_hop_abstract_query_synthesizer
6,Hwo does military service in the US armed forc...,[<1-hop>\n\nPersons on active duty in the U.S....,Persons on active duty in the U.S. Armed Force...,multi_hop_abstract_query_synthesizer
7,How does the retirement of the IRS Data Retrie...,[<1-hop>\n\nApplication and Verification Guide...,The retirement of the IRS Data Retrieval Tool ...,multi_hop_abstract_query_synthesizer
8,how does FA-DDX help students with FAFSA and w...,[<1-hop>\n\nApplication and Verification Guide...,FA-DDX helps students by allowing the transfer...,multi_hop_specific_query_synthesizer
9,How does the ISIR relate to the FAFSA Submissi...,[<1-hop>\n\nOutput Documents After processing ...,The ISIR is an output document produced after ...,multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [37]:
from langsmith import Client

client = Client()

dataset_name = "Loan Synthetic Data"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Loan Synthetic Data"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [38]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [39]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [40]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [41]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [42]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan RAG"
)

In [43]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [44]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

For our LLM, we will be using TogetherAI's endpoints as well!

We're going to be using Meta Llama 3.1 70B Instruct Turbo - a powerful model which should get us powerful results!

In [45]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [46]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [47]:
rag_chain.invoke({"question" : "What kinds of loans are available?"})

'The kinds of loans available mentioned in the context are:\n\n- Direct Subsidized Loan  \n- Direct Unsubsidized Loan  \n- Direct PLUS Loan (student Federal PLUS Loan and parent Direct PLUS Loan)  \n- Subsidized and Unsubsidized Federal Stafford Loans (made under the Federal Family Education Loan (FFEL) Program before July 1, 2010)  \n- Federal SLS Loans  \n- Federal PLUS Loans (made under the FFEL Program before July 1, 2010)  \n\nAdditionally, it is noted that graduate or professional students are eligible only for Direct Unsubsidized Loans (not Direct Subsidized Loans). Parents may receive a Direct PLUS Loan on behalf of a dependent student.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [48]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [49]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

empathy_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "empathy": "Is this response empathetic? Does it make the user feel like they are being heard?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`:
- `labeled_helpfulness_evaluator`:
- `empathy_evaluator`:

#### ✅ Answer :

🧐 qa_evaluator :

🔹 This evaluator checks the factual accuracy of the model’s response. It compares the answer generated by your RAG system to the reference (ground truth) answer provided in your dataset.

In detail:

🔸 It asks: “Did the model provide the correct information?”

🔸 If the answer matches the reference or is factually correct based on the context, it scores well.

🔸 This is the most basic and essential evaluation for any question-answering system, ensuring the model isn’t hallucinating or making mistakes.

🤝 labeled_helpfulness_evaluator :

🔹 This evaluator measures how helpful the model’s answer is to the user, taking into account the reference answer.

In detail:

🔸 It asks: “Does the answer actually help the user solve their problem or understand the topic?”

🔸 It considers not just correctness, but also clarity, completeness, and usefulness.

🔸 For example, an answer might be correct but too brief or vague to be helpful; this evaluator would score such an answer lower.

💖 empathy_evaluator :

🔹 This evaluator assesses the empathy shown in the model’s response. Evaluates the empathy in the response, checking if the answer is understanding, kind, and makes the user feel heard.

In detail:

🔸 It asks: “Does the answer show understanding and care for the user’s situation or feelings?”

🔸 It looks for language that is kind, supportive, and makes the user feel acknowledged.

🔸 For example, an empathetic answer might say, “I understand this can be confusing, but here’s how you can proceed…” rather than just giving a dry fact.

## LangSmith Evaluation

In [50]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'dependable-quilt-26' at:
https://smith.langchain.com/o/9cc120d2-ca66-4b3b-9545-ca0497b68deb/datasets/d2c86547-976c-40f4-a86a-2f4dcba67907/compare?selectedSessions=0c0ee390-8aeb-4163-9a3c-31592d556cbf




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,how do 2024-25 and 2025-26 FAFSA help veterans...,The 2024-25 and 2025-26 FAFSA cycles provide s...,None,The 2024-25 and 2025-26 FAFSA are important fo...,1,1,0,3.242720,e4fdd8c5-3735-47a5-a24f-fa07cd54507f,4e15cdec-bf18-43bb-bc98-2fdcb845ab1b
1,"Based on Chapter 2 examples, how does supporti...","Based on Chapter 2 examples, if a student supp...",None,"According to Chapter 2 examples, if a student ...",1,1,0,1.960681,6b30a221-6151-47a7-a018-0cde76f1f701,9ac2a6ae-11e2-40e3-ba73-214f75ee9f26
2,How does the ISIR relate to the FAFSA Submissi...,The ISIR (Institutional Student Information Re...,None,The ISIR is an output document produced after ...,1,1,0,8.568024,69bc2692-e54a-4a5a-aabf-180c47031234,2d6a90ab-70bd-4fba-8726-191f31a563a1
3,how does FA-DDX help students with FAFSA and w...,The FA-DDX (FUTURE Act Direct Data Exchange) h...,None,FA-DDX helps students by allowing the transfer...,1,1,0,3.351461,350c235e-7426-4108-93f2-446fe7755b6a,0ddc83e3-3dc5-483b-8efe-ec6043105e35
4,How does the retirement of the IRS Data Retrie...,The context does not provide specific informat...,None,The retirement of the IRS Data Retrieval Tool ...,1,0,0,3.652698,dd668a50-7ab0-4d01-8660-0b5437660e86,dacbc0b9-7224-4009-8033-f8cb7e79a979
5,Hwo does military service in the US armed forc...,Persons on active duty in the U.S. Armed Force...,None,Persons on active duty in the U.S. Armed Force...,1,1,0,2.189888,4a821c51-5fa6-41f5-8d27-ad5ba982c9e6,fb009414-e4c7-43ef-b7a1-8ada5f281069
6,How do mailing procedures and address changes ...,"Based on the context provided, the mailing pro...",None,Mailing procedures for FAFSA forms specify tha...,1,1,0,4.270217,d19362c0-7edc-4368-a33c-3b0391dbc1b8,ab94bad1-8a90-421f-ab06-9c37ff8ce9e7
7,How does the IRS data about tax info relate to...,The IRS data about tax information (FTI) can o...,None,"The IRS data, considered FTI, includes informa...",1,1,0,2.623983,3c4474ed-d0a2-4bec-96a0-d3bc429dda74,e3fc1183-72b1-4ab1-a533-440d9a10e460
8,Could you please explain the significance of t...,The 2025-26 FAFSA Specifications Guide is a ke...,None,"The 2025-26 FAFSA Specifications Guide, specif...",1,0,0,4.351658,34a21981-8d6e-4206-a60c-f510d5a2a8d4,d31f3b57-1f08-489c-bcf7-e714c203e074
9,What are the differences between the 2024-25 F...,"Based on the provided context, the differences...",None,There are two active portals corresponding to ...,1,1,0,4.539496,ad672d36-a892-4d12-be6f-a172c491ee6c,fb6fe4bf-c876-4769-9ebc-3169930f0230


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [51]:
EMPATHY_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You must answer the question using empathy and kindness, and make sure the user feels heard.

Context: {context}
Question: {question}
"""

empathy_rag_prompt = ChatPromptTemplate.from_template(EMPATHY_RAG_PROMPT)

In [52]:
rag_documents = docs

In [53]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

#### ✅ Answer :

Changing the chunk size affects how your documents are split into pieces before being embedded and stored in the vector database for retrieval.

Detailed Explanation:

💠 Chunk size determines how much text is in each segment (or "chunk") that your RAG system processes.

💠 If the chunk size is too small:
Each chunk may contain only a small part of the context.
The retriever might miss important information that is spread across multiple chunks.
Answers may be less complete or relevant because the model sees less context at a time.

💠 If the chunk size is too large:
Each chunk may contain too much information, possibly including unrelated or noisy data.
The retriever might return overly broad or less focused context.
Large chunks can also hit token limits for the language model, causing truncation.

In summary:

Modifying the chunk size changes how much context the model has access to for each question. The right chunk size helps the retriever find the most relevant and complete information, improving the accuracy and usefulness of your application's answers.

In [54]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

#### ✅ Answer :

Changing the embedding model can have a big impact on your application's performance because the embedding model determines how text is converted into vectors for similarity search.

🔸 The embedding model is responsible for converting text chunks into numerical vectors that capture the meaning and context of the text.

🔸 When you change the embedding model (for example, from text-embedding-3-small to text-embedding-3-large), you are changing how well the system can understand and represent the information in your documents.

How this affects performance:

🔹 Better embedding models (usually larger or more recent ones like text-embedding-3-large) can capture more subtle meanings, relationships, and context from the text.

🔹 This means that when a user asks a question, the retriever can find more relevant and accurate chunks of information, because the vectors are a better match for the question’s intent.

🔹 Weaker or older embedding models might miss important connections or return less relevant results, leading to less accurate or helpful answers.

So, Upgrading your embedding model can significantly improve how well your application retrieves the right information, making your RAG system more accurate, relevant, and useful for users.

In [55]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Loan Data for RAG"
)

In [56]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [57]:
empathy_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | empathy_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [58]:
empathy_rag_chain.invoke({"question" : "What kinds of loans are available?"})

"Thank you for your question! Based on the information provided in the context, there are several types of loans available for students and parents:\n\n1. **Direct Subsidized Loans** – These are loans available to students based on financial need, and the government pays the interest while the student is in school at least half-time.\n\n2. **Direct Unsubsidized Loans** – These loans are available to students regardless of financial need, and interest accrues while the student is in school.\n\n3. **Direct PLUS Loans** – These loans are available to the parents of dependent undergraduate students to help cover educational costs, and also available to graduate and professional students. The loan amount can cover up to the student's cost of attendance (COA) minus any other financial aid received.\n\nIt's also mentioned that if parents are unable to obtain a Direct PLUS Loan, dependent students may be eligible for additional Direct Unsubsidized Loan amounts.\n\nI hope this helps clarify the

Finally, we can evaluate the new chain on the same test set!

In [59]:
evaluate(
    empathy_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "empathy_rag_chain"},
)

View the evaluation results for experiment: 'reflecting-winter-42' at:
https://smith.langchain.com/o/9cc120d2-ca66-4b3b-9545-ca0497b68deb/datasets/d2c86547-976c-40f4-a86a-2f4dcba67907/compare?selectedSessions=e34b10d4-265f-4f84-8fa7-a23e5683b8e2




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,how do 2024-25 and 2025-26 FAFSA help veterans...,Thank you for your question! It’s wonderful to...,None,The 2024-25 and 2025-26 FAFSA are important fo...,1,1,1,4.044108,e4fdd8c5-3735-47a5-a24f-fa07cd54507f,a54bf21b-e456-4b28-836e-c144a8502588
1,"Based on Chapter 2 examples, how does supporti...",Thank you for your thoughtful question. Based ...,None,"According to Chapter 2 examples, if a student ...",1,1,1,5.601207,6b30a221-6151-47a7-a018-0cde76f1f701,4d73d111-dfb3-4ed0-8ba1-b126ab57d7be
2,How does the ISIR relate to the FAFSA Submissi...,Thank you for your thoughtful question! Based ...,None,The ISIR is an output document produced after ...,1,1,1,5.926643,69bc2692-e54a-4a5a-aabf-180c47031234,f233e1eb-0016-4f6c-b0db-ea4f47e4a19c
3,how does FA-DDX help students with FAFSA and w...,Thank you for your thoughtful question. It sou...,None,FA-DDX helps students by allowing the transfer...,1,1,1,8.874385,350c235e-7426-4108-93f2-446fe7755b6a,abce8a6e-5a5c-4b70-8402-3d8958281d61
4,How does the retirement of the IRS Data Retrie...,Thank you for your thoughtful question. Based ...,None,The retirement of the IRS Data Retrieval Tool ...,1,0,1,12.683052,dd668a50-7ab0-4d01-8660-0b5437660e86,7398bfef-1355-426b-b808-83ffcab1dc14
5,Hwo does military service in the US armed forc...,Thank you for your thoughtful question about h...,None,Persons on active duty in the U.S. Armed Force...,1,1,1,6.213012,4a821c51-5fa6-41f5-8d27-ad5ba982c9e6,153b41bf-24a6-4c83-a004-29ac2c0a71e4
6,How do mailing procedures and address changes ...,Thank you for your thoughtful question. From t...,None,Mailing procedures for FAFSA forms specify tha...,1,1,1,4.371998,d19362c0-7edc-4368-a33c-3b0391dbc1b8,ea2547ec-565c-4bee-8310-60d30f342083
7,How does the IRS data about tax info relate to...,Thank you for your thoughtful question. Based ...,None,"The IRS data, considered FTI, includes informa...",1,1,1,5.993657,3c4474ed-d0a2-4bec-96a0-d3bc429dda74,70faad11-aad1-4412-a2bc-30bcf6026852
8,Could you please explain the significance of t...,Thank you for your thoughtful question. From t...,None,"The 2025-26 FAFSA Specifications Guide, specif...",1,0,1,4.068682,34a21981-8d6e-4206-a60c-f510d5a2a8d4,71e74c17-884f-4314-8868-e24cddfb252d
9,What are the differences between the 2024-25 F...,Thank you for your thoughtful question! It's c...,None,There are two active portals corresponding to ...,1,1,1,5.062609,ad672d36-a892-4d12-be6f-a172c491ee6c,24c57def-e76d-4467-ad9f-c538738cd4fe


#### 


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.

####  ✅ Answer :

Below provided screenshot shows the difference between the two chains.

![Chains Difference](data/Chains_Difference.png)

Original Chain Screenshot(dependable-quilt-26): 

![Original Chain](data/Original_Chain.png)

Dope/Empathy Chain Screenshot(reflecting-winter-42):

![Dope Empathy Chain](data/Dope_Empathy_Chain.png)

Comparison of the Two Chains:

Original Chain (dependable-quilt-26):

🔹 Correctness: 0.9167 (high)

🔹 Helpfulness: 0.75 (moderate to high)

🔹 Empathy: 0.00 (none of the answers were marked as empathetic)

Dope/Empathy Chain (reflecting-winter-42):

🔹 Correctness: 0.9167 (same as before)

🔹 Helpfulness: 0.75 (same as before)

🔹 Empathy: 1.00 (all answers are now marked as empathetic)

🟣 What changed and why:

1. Empathy:

The most significant improvement is in the empathy metric. In the dope/empathy chain, every answer is marked as empathetic (y), thanks to the new prompt that specifically instructs the model to be kind, understanding, and supportive. In the original chain, empathy was not present at all.

2. Helpfulness:

The helpfulness score remained the same (0.75) in both chains. This means that making the answers more empathetic did not negatively impact how useful or supportive the answers were for the user.

3. Correctness:

The correctness score also remained unchanged (0.9167), indicating that the factual accuracy of the answers was maintained even after making the responses more empathetic.


| Metric      | Original Chain | Dope/Empathy Chain | Change & Reason                                      |
|-------------|---------------|--------------------|------------------------------------------------------|
| Correctness | 0.9167        | 0.9167             | No change; factual accuracy stayed high              |
| Helpfulness | 0.75          | 0.75               | No change; answers remained equally helpful          |
| Empathy     | 0.00          | 1.00               | Increased; prompt explicitly asked for empathy       |


🟢 In summary:

🔸 The new "dope" chain made the answers much more empathetic, as intended.

🔸 Helpfulness and correctness were not affected, showing that it’s possible to increase empathy in responses without sacrificing accuracy or usefulness.

🔸 This demonstrates that prompt engineering can improve the conversational quality (like empathy) of your system while maintaining its effectiveness and reliability.